<a href="https://colab.research.google.com/github/kousiknandy/pycolab/blob/main/credit_tracking_service.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
import heapq
import bisect

class creditTracker:
    def __init__(self):
        self.journal = [(-1, 0)]
        self.credits = []

    def add_credit(self, amount, cur_time, exp_time):
        self.flush_expired(cur_time)
        heapq.heappush(self.credits, (exp_time, amount))
        self.journal.append((cur_time, self.journal[-1][1] + amount))

    def flush_expired(self, cur_time):
        while self.credits and self.credits[0][0] < cur_time:
            exp_time, amount = heapq.heappop(self.credits)
            self.journal.append((exp_time, self.journal[-1][1] - amount))

    def use_credit(self, amount, cur_time):
        self.flush_expired(cur_time)
        remaining = sum(a[1] for a in self.credits)
        if remaining < amount: return False
        self.journal.append((cur_time, self.journal[-1][1] - amount))
        while amount:
            exp_time, value = heapq.heappop(self.credits)
            if value > amount:
                heapq.heappush(self.credits, (exp_time, value - amount))
                amount = 0
            else:
                amount -= value
        return True

    def get_balance(self, cur_time):
        self.flush_expired(cur_time)
        print(".  ", self.journal, self.credits)
        i = bisect.bisect(self.journal, cur_time, key=lambda x: x[0])
        return self.journal[i-1][1]

In [34]:
def test_creditTracker():
    tracker = creditTracker()

    # Test initial state
    assert tracker.get_balance(0) == 0
    print("Initial balance test passed.")

    # Test adding credits
    tracker.add_credit(100, 10, 20) # Add 100 at time 10, expires at 20
    assert tracker.get_balance(10) == 100
    print("Add credit (100) test passed.")

    tracker.add_credit(50, 12, 25)  # Add 50 at time 12, expires at 25
    assert tracker.get_balance(12) == 150
    assert tracker.get_balance(15) == 150
    print("Add credit (50) test passed.")

    # Test using credits
    assert tracker.use_credit(70, 15) == True # Use 70 at time 15
    assert tracker.get_balance(15) == 80 # 150 - 70 = 80
    print("Use credit (70) test passed.")

    # Test using more than available
    assert tracker.use_credit(100, 16) == False # Only 80 available (100-70 from first, 50 from second)
    assert tracker.get_balance(16) == 80 # Balance should not change
    print("Use credit (too much) test passed.")

    # Test flush_expired and get_balance after expiration
    assert tracker.get_balance(20) == 80 # First credit (100) was added at time 10, expired at 20. But 70 was used earlier.
                                        # (100-70 from first credit) + 50 from second credit. At time 20, 30 from first credit expired. So 30+50 = 80
    # tracker.flush_expired(21) # First credit part should be expired
    assert tracker.get_balance(21) == 50 # 50 from the second credit should remain
    print("Flush expired test passed.")

    # Test using remaining credits
    assert tracker.use_credit(50, 22) == True
    assert tracker.get_balance(22) == 0
    print("Use remaining credit test passed.")

    # Test adding credit after all expired/used
    tracker.add_credit(200, 30, 40)
    assert tracker.get_balance(30) == 200
    print("Add credit after all used test passed.")

    tracker.flush_expired(41)
    assert tracker.get_balance(41) == 0
    print("Final flush test passed.")

    print("All creditTracker tests passed!")

test_creditTracker()


.   [(-1, 0)] []
Initial balance test passed.
.   [(-1, 0), (10, 100)] [(20, 100)]
Add credit (100) test passed.
.   [(-1, 0), (10, 100), (12, 150)] [(20, 100), (25, 50)]
.   [(-1, 0), (10, 100), (12, 150)] [(20, 100), (25, 50)]
Add credit (50) test passed.
.   [(-1, 0), (10, 100), (12, 150), (15, 80)] [(20, 30), (25, 50)]
Use credit (70) test passed.
.   [(-1, 0), (10, 100), (12, 150), (15, 80)] [(20, 30), (25, 50)]
Use credit (too much) test passed.
.   [(-1, 0), (10, 100), (12, 150), (15, 80)] [(20, 30), (25, 50)]
.   [(-1, 0), (10, 100), (12, 150), (15, 80), (20, 50)] [(25, 50)]
Flush expired test passed.
.   [(-1, 0), (10, 100), (12, 150), (15, 80), (20, 50), (22, 0)] []
Use remaining credit test passed.
.   [(-1, 0), (10, 100), (12, 150), (15, 80), (20, 50), (22, 0), (30, 200)] [(40, 200)]
Add credit after all used test passed.
.   [(-1, 0), (10, 100), (12, 150), (15, 80), (20, 50), (22, 0), (30, 200), (40, 0)] []
Final flush test passed.
All creditTracker tests passed!
